# Inspect Qwen2.5-1.5B-Instruct Configuration

Load and print the checkpoint-specific configuration from Hugging Face.

In [1]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
print(config)

/Users/yudialex/Projects/alignment-experiments/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  

In [2]:
config.num_hidden_layers

28

In [29]:
def parameters(config):
    mha = 4 * config.hidden_size * config.hidden_size
    print(f"mha {mha:.2e}")
    fnn = 3 * config.hidden_size * config.intermediate_size
    print(f"fnn {fnn:.2e}")
    transformer_block = mha + fnn
    transformer = transformer_block * config.num_hidden_layers
    print(f"transformer {transformer:.2e}")
    embedding = config.vocab_size * config.hidden_size
    print(f"embedding {embedding:.2e}")
    output_head = config.hidden_size * config.vocab_size
    print(f"output_head {output_head:.2e}")
    total = transformer + embedding + output_head
    print(f"total {total:.2e}")
    print(f"{total*2:.2e} bytes to load the model")
    return total

In [30]:
parameters(config)

mha 9.44e+06
fnn 4.13e+07
transformer 1.42e+09
embedding 2.33e+08
output_head 2.33e+08
total 1.89e+09
3.77e+09 bytes to load the model


1887043584

In [31]:
def forward(
    config, 
    sequence_length=config.max_position_embeddings, 
    batch_size=1):
    x = batch_size * sequence_length
    embedding = x * config.hidden_size
    mha = 3 * x * config.hidden_size + batch_size * config.hidden_size + batch_size * sequence_length * sequence_length
    print(f"mha {mha:.2e}")
    fnn = 2 * x * config.intermediate_size + x * config.hidden_size
    print(f"fnn {fnn:.2e}")
    transformer_block = mha + fnn
    transformer = transformer_block * config.num_hidden_layers
    print(f"transformer {transformer:.2e}")
    output_head = x * config.vocab_size
    print(f"output_head {output_head:.2e}")
    total = transformer + embedding + output_head
    print(f"total {total:.2e}")
    print(f"{total*2:.2e} bytes to run forward on the model")
    return total
          
    
    
    

In [32]:
config.max_position_embeddings

32768

In [33]:
forward(config, sequence_length=1024, batch_size=32)

mha 1.85e+08
fnn 6.38e+08
transformer 2.30e+10
output_head 4.98e+09
total 2.80e+10
5.61e+10 bytes to run forward on the model


28048687104

In [39]:
def backward(config, sequence_length=1024, batch_size=32):
    params = parameters(config)
    grads = parameters(config)
    momentums = 2 * parameters(config)
    activations = forward(config, sequence_length=sequence_length, batch_size=batch_size)
    total = params+grads+momentums+activations
    print(f">>> {total*2:.2e} total memory to one training pass at {sequence_length} and {batch_size} ")
    return total
    

In [41]:
backward(config, sequence_length=30000, batch_size=1)

mha 9.44e+06
fnn 4.13e+07
transformer 1.42e+09
embedding 2.33e+08
output_head 2.33e+08
total 1.89e+09
3.77e+09 bytes to load the model
mha 9.44e+06
fnn 4.13e+07
transformer 1.42e+09
embedding 2.33e+08
output_head 2.33e+08
total 1.89e+09
3.77e+09 bytes to load the model
mha 9.44e+06
fnn 4.13e+07
transformer 1.42e+09
embedding 2.33e+08
output_head 2.33e+08
total 1.89e+09
3.77e+09 bytes to load the model
mha 1.04e+09
fnn 5.84e+08
transformer 4.54e+10
output_head 4.56e+09
total 5.00e+10
1.00e+11 bytes to run forward on the model
>>> 1.15e+11 total memory to one training pass at 30000 and 1 


57566137344

Table 6. Default LoRA Fine-tuning Parameters
Parameter Value
Batch Size 2
Gradient Accumulation Steps 8
Warm-up Steps 5
Learning Rate (LR) Optimiser LR scheduling Weight Decay 1e-5
adamw 8bit
linear
0.01
Rank Alpha LoRA Dropout 32
64
0.0

In [50]:
def params_LoRA(config, rank):
    mha = 4 * 2 * (config.hidden_size * rank)
    print(f"mha {mha:.2e}")
    fnn = 3 * (config.hidden_size * rank + rank * config.intermediate_size) 
    print(f"fnn {fnn:.2e}")
    transformer_block = mha + fnn
    transformer = transformer_block * config.num_hidden_layers
    print(f"transformer {transformer:.2e}")
    total = transformer 
    print(f"total {total:.2e}")
    print(f"{total*2:.2e} bytes to load the model")
    return total

In [51]:
params_LoRA(config, 32)

mha 3.93e+05
fnn 1.01e+06
transformer 3.92e+07
total 3.92e+07
7.84e+07 bytes to load the model


39223296

In [52]:
parameters(config)

mha 9.44e+06
fnn 4.13e+07
transformer 1.42e+09
embedding 2.33e+08
output_head 2.33e+08
total 1.89e+09
3.77e+09 bytes to load the model


1887043584

In [57]:
def full_LoRA_finetuning(config, sequence_length=1024, batch_size=2, rank=32):
    p = parameters(config)
    pl = params_LoRA(config, rank)
    activations = forward(config, sequence_length, batch_size)
    grad = params_LoRA(config, rank)
    momentums = 2 *  params_LoRA(config, rank)
    total = p+pl+grad+momentums+activations
    print(f">>> {total*2:.2e} total memory to one training pass at {sequence_length} and {batch_size} ")
    return total
    

In [59]:
full_LoRA_finetuning(config, sequence_length=2048)

mha 9.44e+06
fnn 4.13e+07
transformer 1.42e+09
embedding 2.33e+08
output_head 2.33e+08
total 1.89e+09
3.77e+09 bytes to load the model
mha 3.93e+05
fnn 1.01e+06
transformer 3.92e+07
total 3.92e+07
7.84e+07 bytes to load the model
mha 2.73e+07
fnn 7.97e+07
transformer 2.99e+09
output_head 6.22e+08
total 3.62e+09
7.25e+09 bytes to run forward on the model
mha 3.93e+05
fnn 1.01e+06
transformer 3.92e+07
total 3.92e+07
7.84e+07 bytes to load the model
mha 3.93e+05
fnn 1.01e+06
transformer 3.92e+07
total 3.92e+07
7.84e+07 bytes to load the model
>>> 1.13e+10 total memory to one training pass at 2048 and 2 


5667377152

In [64]:
tokens_per_dataset = 1e6

In [65]:
params = 1.5e9

In [66]:
flops = 7 * tokens_per_dataset * params

In [67]:
flops

1.05e+16

In [68]:
GPU_flops_per_second = 100 * 1e12

In [69]:
flops / GPU_flops_per_second

105.0